# TopDrive AI — 02: Training v3 — DANN + Conformal Prediction
**Closes the sim-to-real gap at training time. Produces calibrated uncertainty at inference time.**

### What's new in v3 (research-backed)
| Component | Change | Evidence |
|---|---|---|
| **DANN** | Gradient reversal domain discriminator on InceptionTime backbone | +10.2pp accuracy on 9-class digital-twin → real transfer (arXiv 2505.21046) |
| **AdaBN** | Per-machine batch-norm stat replacement at inference | Simple, stable; no extra training required |
| **L2-SP regularisation** | Penalise weight drift from synthetic-pretrained init during real-data fine-tune | Consistently beats L2 and layer-freezing across domain settings |
| **Conformal prediction** | Distribution-free prediction sets with 90% coverage guarantee | Finite-sample, no normality assumption — valid even under sim→real distribution shift |
| **Tiered inference** | Auto-classify / Alert / Abstain based on ensemble disagreement + conformal set size | Enables safe deployment with known per-tier reliability |


## 1. Environment & Config

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free',
                    '--format=csv,noheader'], capture_output=True, text=True)
print("GPU:", r.stdout.strip() or "CPU only")


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


In [ ]:
from pathlib import Path
import torch

DRIVE_ROOT  = Path('/content/drive/MyDrive')
REPO_DIR    = DRIVE_ROOT / 'topdrive_ai' / 'plc_simulation'
DATA_DIR    = DRIVE_ROOT / 'topdrive_ai' / 'datasets' / 'synthetic_v3'
MODEL_DIR   = DRIVE_ROOT / 'topdrive_ai' / 'models_v3'
REAL_DIR    = None   # ← set to real captures dir when available
                     # e.g. DRIVE_ROOT / 'topdrive_ai' / 'real_captures'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_DIR))

# ── Hyperparameters ────────────────────────────────────────────────────────
WINDOW_SIZE     = 1000       # 10s at 100Hz
STRIDE          = 200        # 80% overlap
FAULT_THRESHOLD = 0.05
BATCH_SIZE      = 72
N_EPOCHS        = 150
LR              = 2e-3
ENSEMBLE_SIZE   = 5
SEED            = 42
FOCAL_GAMMA     = 2.0
N_TTA           = 8
N_CLASSES       = 9

# DANN-specific
DANN_LAMBDA_MAX = 0.5    # max GRL lambda (anneals from 0 → 0.5)
DANN_WARMUP     = 20     # epochs before GRL switches on

# L2-SP (for fine-tuning on real data)
L2SP_ALPHA      = 0.01   # weight on L2-SP term

# Conformal prediction
CP_ALPHA        = 0.10   # target miscoverage rate — 90% coverage guarantee

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Dependencies

In [ ]:
%%capture
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install scikit-learn pandas pyarrow numpy tqdm matplotlib seaborn --quiet

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}  CUDA: {torch.cuda.is_available()}')

## 3. Load & Validate Manifest

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from generate_dataset import load_manifest, validate_manifest, FAULT_CLASSES, CLASS_NAMES

manifest = load_manifest(DATA_DIR)
print(f"Manifest: {len(manifest)} scenarios  |  {manifest['fault_class'].nunique()} classes")
validate_manifest(manifest)
print()
dist = manifest.groupby(['fault_class','scenario_type']).size()
for (fc,st), n in dist.items():
    bar = '█' * (n // 50)
    print(f"  [{fc}] {st:<20s} {n:5d}  {bar}")


## 4. Splits

In [ ]:
from dataset import create_splits

train_ids, val_ids, test_ids = create_splits(
    manifest, class_map=FAULT_CLASSES,
    split_ratio=(0.70, 0.15, 0.15), seed=SEED,
)
# Reserve a calibration subset from val for conformal prediction
# (must NOT be used during model selection)
cal_size    = min(500, len(val_ids) // 2)
cal_ids     = val_ids[:cal_size]
val_ids_fit = val_ids[cal_size:]

print(f"train={len(train_ids)}  val={len(val_ids_fit)}  cal={len(cal_ids)}  test={len(test_ids)}")


## 5. Inference Channel List

In [ ]:
INFERENCE_CHANNELS = [
    'torque_ftlbs','rpm','turns','pressure',
    'torque_gradient','rpm_collapse_rate','torque_osc_index',
    'norm_torque','power_proxy','torque_efficiency',
]
N_CHANNELS = len(INFERENCE_CHANNELS)
print(f"Feature channels ({N_CHANNELS}): {INFERENCE_CHANNELS}")


## 6. Dataset — Online Augmentation + Unlabeled Domain Data

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from dataset import build_dataset_index

class AugmentedTopDriveDataset(Dataset):
    """Augmented dataset for labeled synthetic data."""
    def __init__(self, X, y, augment=False, rng_seed=None):
        self.X       = torch.from_numpy(X.astype(np.float32))
        self.y       = torch.from_numpy(y.astype(np.int64))
        self.augment = augment
        self.rng     = np.random.RandomState(rng_seed)
        self._cls_idx = {c: np.where(y == c)[0] for c in range(N_CLASSES)}

    def __len__(self): return len(self.y)

    def _augment(self, x, y_int):
        gain  = torch.empty(x.shape[0], 1).uniform_(0.92, 1.08)
        x     = x * gain + torch.randn_like(x) * x.std(dim=1, keepdim=True) * 0.015
        shift = self.rng.randint(-50, 51)
        if shift > 0:
            x = torch.cat([x[:, :shift].flip(1), x[:, :-shift]], dim=1)
        elif shift < 0:
            x = torch.cat([x[:, -shift:], x[:, shift:].flip(1)], dim=1)
        if self.rng.random() < 0.10:
            x[self.rng.randint(2, x.shape[0])] = 0.0
        partners = self._cls_idx.get(y_int, [])
        if len(partners) > 1 and self.rng.random() < 0.3:
            lam = float(np.random.beta(0.2, 0.2))
            x   = lam * x + (1 - lam) * self.X[self.rng.choice(partners)]
        return x

    def __getitem__(self, i):
        x = self.X[i].clone()
        y = self.y[i]
        if self.augment:
            x = self._augment(x, int(y))
        return x, y


class UnlabeledDomainDataset(Dataset):
    """
    Unlabeled real rig data for DANN domain alignment.
    If REAL_DIR is None, uses a held-out synthetic subset as
    a proxy (less effective but allows training without real data).
    """
    def __init__(self, X):
        self.X = torch.from_numpy(X.astype(np.float32))
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], torch.tensor(-1)  # -1 = no label


print("Building window indices...")
SENSOR_DIR = DATA_DIR / 'sensors'

Xtr, ytr, _ = build_dataset_index(manifest, train_ids, SENSOR_DIR, FAULT_CLASSES,
    window_size=WINDOW_SIZE, stride=STRIDE, fault_threshold=FAULT_THRESHOLD,
    channels_mode='inference')
Xva, yva, _ = build_dataset_index(manifest, val_ids_fit, SENSOR_DIR, FAULT_CLASSES,
    window_size=WINDOW_SIZE, stride=STRIDE, fault_threshold=FAULT_THRESHOLD,
    channels_mode='inference')
Xca, yca, _ = build_dataset_index(manifest, cal_ids, SENSOR_DIR, FAULT_CLASSES,
    window_size=WINDOW_SIZE, stride=STRIDE, fault_threshold=FAULT_THRESHOLD,
    channels_mode='inference')
Xte, yte, _ = build_dataset_index(manifest, test_ids, SENSOR_DIR, FAULT_CLASSES,
    window_size=WINDOW_SIZE, stride=STRIDE, fault_threshold=FAULT_THRESHOLD,
    channels_mode='inference')

for name, y in [('train',ytr),('val',yva),('cal',yca),('test',yte)]:
    dist = Counter(int(l) for l in y)
    miss = [c for c in range(9) if dist.get(c,0) == 0]
    assert not miss, f"ABORT: missing {[CLASS_NAMES[c] for c in miss]} in {name}"
    print(f"  {name}: ✓  n={sum(dist.values())}  min_cls={min(dist.values())}")


In [ ]:
# Stack and normalise
def stack(Xl, yl):
    X = np.concatenate(Xl, 0).transpose(0,2,1).astype(np.float32)
    y = yl.astype(np.int64)
    return X, y

Xtr, ytr = stack(Xtr, ytr)
Xva, yva = stack(Xva, yva)
Xca, yca = stack(Xca, yca)
Xte, yte = stack(Xte, yte)

# Z-score normalised on train
mean = Xtr.mean(axis=(0,2), keepdims=True)
std  = Xtr.std( axis=(0,2), keepdims=True) + 1e-8
Xtr  = (Xtr - mean) / std
Xva  = (Xva - mean) / std
Xca  = (Xca - mean) / std
Xte  = (Xte - mean) / std

import pickle
norm_stats = {'mean': mean, 'std': std}
with open(MODEL_DIR / 'norm_stats_v3.pkl', 'wb') as f:
    pickle.dump(norm_stats, f)

n_channels = Xtr.shape[1]
seq_len    = Xtr.shape[2]
print(f"Shape: train={Xtr.shape}  val={Xva.shape}  cal={Xca.shape}  test={Xte.shape}")


In [ ]:
# Unlabeled domain data — use held-out synthetic subset as proxy if no real data
if REAL_DIR is not None and Path(REAL_DIR).exists():
    print("Loading real unlabeled data from REAL_DIR...")
    # Load all parquet files in REAL_DIR — no labels needed
    real_files = list(Path(REAL_DIR).glob('*.parquet'))
    real_windows = []
    for f in real_files:
        df   = pd.read_parquet(f)
        avail = [c for c in INFERENCE_CHANNELS if c in df.columns]
        if len(avail) < N_CHANNELS:
            continue
        data = df[INFERENCE_CHANNELS].values.astype(np.float32)
        # Slide windows
        for start in range(0, len(data) - WINDOW_SIZE, STRIDE):
            real_windows.append(data[start:start+WINDOW_SIZE].T)
    X_unlabeled = np.stack(real_windows, 0)
    # Normalise with same stats
    X_unlabeled = (X_unlabeled - mean) / std
    print(f"  Real unlabeled windows: {len(X_unlabeled)}")
else:
    # Use 20% of synthetic test data as domain proxy
    # Less effective than real data, but lets DANN run during development
    n_proxy     = max(500, len(Xte) // 5)
    X_unlabeled = Xte[:n_proxy].copy()
    print(f"No real data — using {n_proxy} synthetic proxy windows for DANN domain head")

# Dataloaders
train_ds   = AugmentedTopDriveDataset(Xtr, ytr, augment=True,  rng_seed=SEED)
val_ds     = AugmentedTopDriveDataset(Xva, yva, augment=False)
cal_ds     = AugmentedTopDriveDataset(Xca, yca, augment=False)
test_ds    = AugmentedTopDriveDataset(Xte, yte, augment=False)
domain_ds  = UnlabeledDomainDataset(X_unlabeled)

class_counts   = np.bincount(ytr, minlength=9)
sample_weights = (1.0 / (class_counts + 1e-8))[ytr]
sampler = WeightedRandomSampler(torch.from_numpy(sample_weights).float(),
                                 num_samples=len(train_ds), replacement=True)

train_loader  = DataLoader(train_ds,  batch_size=BATCH_SIZE, sampler=sampler,
                           num_workers=2, pin_memory=True, persistent_workers=True)
val_loader    = DataLoader(val_ds,    batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
cal_loader    = DataLoader(cal_ds,    batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_ds,   batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
domain_loader = DataLoader(domain_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, persistent_workers=True,
                           drop_last=True)

print(f"train: {len(train_ds)} samples  domain (DANN): {len(domain_ds)} samples")


## 7. Model: Multi-Scale InceptionTime + DANN Discriminator

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function


# ─────────────────────────────────────────────────────────────────────────────
# GRADIENT REVERSAL LAYER
#
# Analogy: it's a two-player game where the feature extractor tries to fool a
# domain discriminator, while also correctly classifying faults. The GRL is
# the mechanism that turns this into a single differentiable system.
# Forward pass: identity. Backward pass: multiply gradient by -λ.
# Result: the backbone is simultaneously pulled toward fault-discriminative
# AND domain-invariant representations.
#
# Reference: Ganin & Lempitsky, JMLR 2016.
# ─────────────────────────────────────────────────────────────────────────────
class GRL(Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


def grad_reverse(x, lambda_):
    return GRL.apply(x, lambda_)


# ─────────────────────────────────────────────────────────────────────────────
# Building blocks (unchanged from v2)
# ─────────────────────────────────────────────────────────────────────────────
class SqueezeExcite(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        s = self.fc(x.mean(dim=2)).unsqueeze(2)
        return x * s


class InceptionBlock(nn.Module):
    def __init__(self, in_ch, n_filters=64, kernel_sizes=(9,19,39), use_se=True):
        super().__init__()
        self.bottleneck = nn.Conv1d(in_ch, n_filters, 1, bias=False)
        self.convs = nn.ModuleList([
            nn.Conv1d(n_filters, n_filters, k, padding=k//2, bias=False)
            for k in kernel_sizes])
        self.mp_conv = nn.Sequential(
            nn.MaxPool1d(3, stride=1, padding=1),
            nn.Conv1d(in_ch, n_filters, 1, bias=False),
        )
        out_ch     = n_filters * (len(kernel_sizes) + 1)
        self.bn    = nn.BatchNorm1d(out_ch)
        self.act   = nn.GELU()
        self.se    = SqueezeExcite(out_ch) if use_se else nn.Identity()

    def forward(self, x):
        bot      = self.bottleneck(x)
        branches = [c(bot) for c in self.convs] + [self.mp_conv(x)]
        return self.se(self.act(self.bn(torch.cat(branches, 1))))


class TemporalAttentionPool(nn.Module):
    def __init__(self, d_model, n_heads=4):
        super().__init__()
        self.attn      = nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=0.1)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        x   = x.transpose(1, 2)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        q   = torch.cat([cls, x], dim=1)
        out, _ = self.attn(q, x, x)
        return out[:, 0]


class DeepInceptionEncoder(nn.Module):
    def __init__(self, in_ch, n_filters=64, depth=6):
        super().__init__()
        out_ch = n_filters * 4
        blocks = []
        ch     = in_ch
        for i in range(depth):
            blocks.append(InceptionBlock(ch, n_filters=n_filters, use_se=(i%2==1)))
            ch = out_ch
        self.blocks   = nn.ModuleList(blocks)
        self.residual = nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, bias=False),
                                       nn.BatchNorm1d(out_ch))
        self.pool     = TemporalAttentionPool(out_ch, n_heads=4)
        self.d_out    = out_ch

    def forward(self, x):
        res = self.residual(x)
        for blk in self.blocks:
            x = blk(x)
        return self.pool(x + res)


# ─────────────────────────────────────────────────────────────────────────────
# Multi-Scale Model with integrated DANN head
#
# Architecture:
#   3 temporal-scale encoders → concat → [fault classifier]
#                                       → [domain discriminator (GRL)]
#
# The domain discriminator outputs a binary logit: 0=synthetic, 1=real.
# During training, GRL ensures the backbone MAXIMISES domain discriminator
# loss (i.e., learns domain-invariant features) while MINIMISING fault
# classification loss.
# ─────────────────────────────────────────────────────────────────────────────
class MultiScaleDANNModel(nn.Module):
    def __init__(self, in_ch, n_classes, n_filters=64, depth=6):
        super().__init__()
        # Three temporal-scale encoders
        self.enc_A = DeepInceptionEncoder(in_ch, n_filters, depth)   # 100Hz
        self.enc_B = DeepInceptionEncoder(in_ch, n_filters, depth)   # 50Hz
        self.enc_C = DeepInceptionEncoder(in_ch, n_filters, depth)   # 25Hz
        fused_dim  = self.enc_A.d_out * 3

        # Fault classifier head
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Dropout(0.3),
            nn.Linear(fused_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, n_classes),
        )

        # Domain discriminator head (binary: synthetic=0 vs real=1)
        # Deeper than strictly necessary — needs to be competitive with encoder
        self.domain_disc = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2),   # 2 classes: synthetic / real
        )

    def _down(self, x, stride):
        return F.avg_pool1d(x, stride, stride) if stride > 1 else x

    def encode(self, x):
        """Forward pass to fused embedding — used by both heads and diagnostics."""
        eA = self.enc_A(self._down(x, 1))
        eB = self.enc_B(self._down(x, 2))
        eC = self.enc_C(self._down(x, 4))
        return torch.cat([eA, eB, eC], dim=1)

    def forward(self, x, grl_lambda=0.0):
        """
        Returns (fault_logits, domain_logits).
        grl_lambda controls domain adaptation strength:
          0.0 = no adaptation (disabled during warmup)
          0.5 = full adaptation (after warmup)
        """
        emb           = self.encode(x)
        fault_logits  = self.classifier(emb)
        domain_emb    = grad_reverse(emb, grl_lambda)
        domain_logits = self.domain_disc(domain_emb)
        return fault_logits, domain_logits

    def predict(self, x):
        """Inference — only returns fault logits."""
        return self.classifier(self.encode(x))

    def embed(self, x):
        return self.encode(x)


# ─────────────────────────────────────────────────────────────────────────────
# Ensemble: 5 independent DANN models
# ─────────────────────────────────────────────────────────────────────────────
class DANNEnsemble(nn.Module):
    def __init__(self, n_members, in_ch, n_classes, n_filters=64, depth=6):
        super().__init__()
        self.members = nn.ModuleList([
            MultiScaleDANNModel(in_ch, n_classes, n_filters, depth)
            for _ in range(n_members)
        ])

    def forward(self, x, grl_lambda=0.0):
        """Returns avg fault logits + list of domain logits (one per member)."""
        f_logits_list  = []
        d_logits_list  = []
        for m in self.members:
            fl, dl = m(x, grl_lambda)
            f_logits_list.append(fl)
            d_logits_list.append(dl)
        return torch.stack(f_logits_list).mean(0), d_logits_list

    def predict(self, x):
        return torch.stack([m.predict(x) for m in self.members]).mean(0)

    def predict_all(self, x):
        return torch.stack([m.predict(x) for m in self.members])

    def diversity_loss(self, x):
        all_p = torch.stack([F.softmax(m.predict(x), 1) for m in self.members])
        M = all_p.shape[0]
        div, n_pairs = 0.0, 0
        for i in range(M):
            for j in range(i+1, M):
                di = all_p[i] - all_p[i].mean(0, keepdim=True)
                dj = all_p[j] - all_p[j].mean(0, keepdim=True)
                corr = (di * dj).sum(0) / (di.norm(dim=0) * dj.norm(dim=0) + 1e-8)
                div += corr.abs().mean(); n_pairs += 1
        return div / (n_pairs + 1e-8)


torch.manual_seed(SEED)
model = DANNEnsemble(ENSEMBLE_SIZE, n_channels, N_CLASSES, n_filters=64, depth=6).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters : {n_params:,}")
dummy = torch.zeros(2, n_channels, seq_len).to(DEVICE)
fl, dl = model(dummy, grl_lambda=0.0)
print(f"Fault logits : {fl.shape}   Domain logits: {dl[0].shape}")


## 8. Loss Functions

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, class_weights=None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer('w',
            torch.tensor(class_weights, dtype=torch.float32)
            if class_weights is not None else None)

    def forward(self, logits, target):
        log_p  = F.log_softmax(logits, dim=1)
        p_t    = log_p.exp().gather(1, target.unsqueeze(1)).squeeze(1)
        focal  = -((1 - p_t) ** self.gamma) * log_p.gather(1, target.unsqueeze(1)).squeeze(1)
        if self.w is not None:
            focal = focal * self.w[target]
        return focal.mean()


class L2SPLoss(nn.Module):
    """
    L2-SP Regularisation — penalises weight drift from a saved checkpoint.
    Used when fine-tuning on real data to prevent catastrophic forgetting
    of patterns learned from synthetic pretraining.

    L_total = L_task + (alpha/2) * ||w - w_0||^2
    where w_0 is the synthetic-pretrained weight snapshot.

    Reference: Li et al., ICML 2018 — consistently beats L2 and layer-freezing.
    """
    def __init__(self, model, alpha=0.01):
        super().__init__()
        self.alpha = alpha
        # Snapshot the current (synthetic-pretrained) weights
        self.w0 = {n: p.detach().clone() for n, p in model.named_parameters()
                   if p.requires_grad}

    def forward(self, model):
        penalty = sum(
            ((p - self.w0[n]) ** 2).sum()
            for n, p in model.named_parameters()
            if n in self.w0 and p.requires_grad
        )
        return self.alpha / 2 * penalty

    def update_anchor(self, model):
        """Call after each fine-tuning phase to update the reference weights."""
        self.w0 = {n: p.detach().clone() for n, p in model.named_parameters()
                   if p.requires_grad}


class_counts = np.bincount(ytr, minlength=9)
inv_freq     = 1.0 / (class_counts + 1e-8)
inv_freq     = inv_freq / inv_freq.sum() * N_CLASSES

fault_criterion  = FocalLoss(gamma=FOCAL_GAMMA, class_weights=inv_freq).to(DEVICE)
domain_criterion = nn.CrossEntropyLoss()
# L2-SP is initialised AFTER the first synthetic training run (Phase 2 / fine-tuning)
print("Loss functions ready.")
print(f"  Focal loss class weights: {np.round(inv_freq, 3)}")


## 9. Training Loop — Synthetic Pretraining + DANN

In [ ]:
import time
from sklearn.metrics import f1_score

optimiser = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimiser, T_0=30, T_mult=2, eta_min=1e-6)

DIVERSITY_WEIGHT = 0.05
domain_iter      = iter(domain_loader)

def evaluate(loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model.predict(xb.to(DEVICE))
            preds.extend(logits.argmax(1).cpu().numpy())
            labels.extend(yb.numpy())
    macro = f1_score(labels, preds, average='macro', zero_division=0)
    per   = f1_score(labels, preds, average=None, zero_division=0, labels=list(range(9)))
    return macro, per


def grl_lambda_schedule(epoch, warmup=DANN_WARMUP, max_lambda=DANN_LAMBDA_MAX):
    """
    Annealing schedule from Ganin & Lempitsky (2016):
    λ = λ_max · (2/(1+exp(−10p)) − 1)   where p = (epoch−warmup)/total
    Starts at 0 (pure classification), ramps to λ_max.
    """
    if epoch < warmup:
        return 0.0
    p = (epoch - warmup) / (N_EPOCHS - warmup)
    return max_lambda * (2.0 / (1.0 + np.exp(-10 * p)) - 1.0)


best_val_f1  = 0.0
ckpt_path    = MODEL_DIR / 'best_ensemble_v3.pt'
history      = {'loss': [], 'dann_loss': [], 'val_f1': [], 'per_class': []}
domain_labels_syn  = torch.zeros(BATCH_SIZE, dtype=torch.long).to(DEVICE)   # 0=synthetic
domain_labels_real = torch.ones( BATCH_SIZE, dtype=torch.long).to(DEVICE)   # 1=real

print(f"Training: {N_EPOCHS} epochs  |  DANN warmup: {DANN_WARMUP} epochs  |  λ_max: {DANN_LAMBDA_MAX}")
print(f"Fault loss: FocalLoss(γ={FOCAL_GAMMA})  |  Diversity weight: {DIVERSITY_WEIGHT}")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    t0     = time.time()
    tot_fl = tot_dl = tot_div = nb_batches = 0

    for (xb, yb) in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        lam    = grl_lambda_schedule(epoch)

        # ── Fault classification loss (synthetic labeled batch) ────────────
        fault_logits, d_logits_syn = model(xb, grl_lambda=lam)
        fl = fault_criterion(fault_logits, yb)

        # ── Domain discriminator loss ──────────────────────────────────────
        # Source (synthetic) side — label 0
        d_labels_s   = domain_labels_syn[:len(xb)]
        d_loss_syn   = sum(domain_criterion(dl, d_labels_s)
                           for dl in d_logits_syn) / len(d_logits_syn)

        # Target (real / proxy) side — label 1
        try:
            xt, _ = next(domain_iter)
        except StopIteration:
            domain_iter = iter(domain_loader)
            xt, _ = next(domain_iter)
        xt = xt.to(DEVICE)
        _, d_logits_real = model(xt, grl_lambda=lam)
        d_labels_r       = domain_labels_real[:len(xt)]
        d_loss_real      = sum(domain_criterion(dl, d_labels_r)
                               for dl in d_logits_real) / len(d_logits_real)

        dann_loss = (d_loss_syn + d_loss_real) * 0.5

        # ── Diversity regularisation (every 5 batches) ───────────────────
        div_loss = model.diversity_loss(xb) * DIVERSITY_WEIGHT if nb_batches % 5 == 0                    else torch.tensor(0.0, device=DEVICE)

        total_loss = fl + dann_loss + div_loss
        optimiser.zero_grad(); total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()

        tot_fl  += fl.item(); tot_dl += dann_loss.item()
        tot_div += div_loss.item(); nb_batches += 1

    scheduler.step()
    val_f1, pcf = evaluate(val_loader)
    history['loss'].append(tot_fl / nb_batches)
    history['dann_loss'].append(tot_dl / nb_batches)
    history['val_f1'].append(val_f1)
    history['per_class'].append(pcf.tolist())

    flag = ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_f1': val_f1, 'per_class_f1': pcf.tolist(),
                    'norm_stats': norm_stats, 'class_map': FAULT_CLASSES,
                    'n_channels': n_channels, 'seq_len': seq_len},
                   ckpt_path)
        flag = ' ← best'

    if epoch % 5 == 0 or epoch <= 3:
        lam_now = grl_lambda_schedule(epoch)
        lr_now  = optimiser.param_groups[0]['lr']
        print(f"Ep {epoch:3d}/{N_EPOCHS} | "
              f"fl {tot_fl/nb_batches:.4f} dann {tot_dl/nb_batches:.4f} | "
              f"val_F1 {val_f1:.4f} min {pcf.min():.4f} | "
              f"λ={lam_now:.3f} lr={lr_now:.2e} | {time.time()-t0:.1f}s{flag}")


## 10. Optional: Fine-Tune on Real Labeled Data (L2-SP)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# L2-SP FINE-TUNING — run this section once you have real labeled fault data
#
# Protocol:
#  1. Load the best synthetic-pretrained checkpoint
#  2. Snapshot weights as the L2-SP anchor (w₀)
#  3. Fine-tune with small learning rate + L2-SP penalty
#  4. Mix 5% synthetic replay data into each batch to prevent forgetting
#
# This is the highest-ROI step after first live data capture with Steve.
# Expected: +10–20pp F1 improvement from adaptation to real rig characteristics.
# ─────────────────────────────────────────────────────────────────────────────

REAL_LABELED_DIR = None   # ← set when labeled captures are available

if REAL_LABELED_DIR is not None and Path(REAL_LABELED_DIR).exists():
    print("=== L2-SP FINE-TUNING ON REAL DATA ===")

    # Load best checkpoint
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    print(f"  Loaded checkpoint: epoch {ckpt['epoch']}  val_F1={ckpt['val_f1']:.4f}")

    # Initialise L2-SP regulariser — anchors to current (synthetic-pretrained) weights
    l2sp = L2SPLoss(model, alpha=L2SP_ALPHA)

    # Build real labeled dataloaders (user must provide labeled parquet files
    # with a 'fault_class' column matching CLASS_NAMES taxonomy)
    # ... (populate X_real_train, y_real_train from REAL_LABELED_DIR)
    # fine_loader = DataLoader(AugmentedTopDriveDataset(X_real_train, y_real_train, augment=True),
    #                          batch_size=32, shuffle=True)

    FT_EPOCHS = 30
    ft_opt    = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    ft_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(ft_opt, T_max=FT_EPOCHS, eta_min=1e-6)

    best_ft_f1 = 0.0
    ft_ckpt    = MODEL_DIR / 'finetuned_real_v3.pt'

    # for epoch in range(1, FT_EPOCHS + 1):
    #     model.train()
    #     for xb, yb in fine_loader:
    #         xb, yb = xb.to(DEVICE), yb.to(DEVICE)
    #         logits = model.predict(xb)
    #         loss   = fault_criterion(logits, yb) + l2sp(model)
    #         ft_opt.zero_grad(); loss.backward()
    #         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    #         ft_opt.step()
    #     ft_sched.step()
    #     ft_f1, _ = evaluate(val_loader)
    #     if ft_f1 > best_ft_f1:
    #         best_ft_f1 = ft_f1
    #         torch.save(model.state_dict(), ft_ckpt)
    #     if epoch % 5 == 0:
    #         print(f"  FT epoch {epoch}/{FT_EPOCHS}  F1={ft_f1:.4f}")

    print("  L2-SP fine-tuning template ready — populate fine_loader and uncomment loop")
else:
    print("No real labeled data — skipping fine-tuning.")
    print("This section will auto-activate once REAL_LABELED_DIR is set.")


## 11. AdaBN: Per-Machine Batch Norm Adaptation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ADAPTIVE BATCH NORMALISATION (AdaBN)
#
# One of the simplest and most reliable unsupervised domain adaptation methods:
# replace ALL batch norm running_mean / running_var statistics with those
# computed from the target domain, keeping all other weights frozen.
#
# Why it works: BN statistics encode domain-specific mean and variance of
# feature activations. Synthetic training produces BN stats calibrated to
# simulator statistics. AdaBN re-calibrates them to each rig's characteristics
# with just a forward pass over unlabeled data — no backward pass required.
#
# Cost: <2 seconds per machine. Run once before deployment on each rig.
# ─────────────────────────────────────────────────────────────────────────────

def apply_adabn(model, target_loader, device, n_batches=50):
    """
    Re-calibrate batch norm statistics to target domain.
    model  : trained DANNEnsemble
    loader : DataLoader of target-domain unlabeled windows
    Returns a copy of model with updated BN stats (does not modify in-place).
    """
    import copy
    adapted = copy.deepcopy(model)

    # Switch BN layers to training mode (computes running stats from current batch)
    # but freeze all other layers
    for m in adapted.modules():
        if isinstance(m, nn.BatchNorm1d):
            m.train()
            m.reset_running_stats()
        else:
            m.eval()

    adapted.to(device)
    with torch.no_grad():
        for i, (xb, _) in enumerate(target_loader):
            if i >= n_batches: break
            adapted(xb.to(device), grl_lambda=0.0)   # forward-only, updates BN running stats

    adapted.eval()
    return adapted


# Test AdaBN with domain proxy data
model.eval()
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])

print("Applying AdaBN with domain proxy data...")
model_adabn = apply_adabn(model, domain_loader, DEVICE, n_batches=50)
val_f1_base, _   = evaluate(val_loader)   # uses original BN stats

# Temporarily swap to adabn model for val eval
orig_model = model
model       = model_adabn
val_f1_adabn, _ = evaluate(val_loader)
model           = orig_model

print(f"Val F1 — original BN : {val_f1_base:.4f}")
print(f"Val F1 — AdaBN       : {val_f1_adabn:.4f}  (delta: {val_f1_adabn - val_f1_base:+.4f})")
print("Note: AdaBN delta on synthetic val is small; real benefit shows on real rig data.")


## 12. TTA + Inference

In [ ]:
def augment_batch(x):
    gain  = torch.empty(x.shape[0], x.shape[1], 1, device=x.device).uniform_(0.95, 1.05)
    noise = torch.randn_like(x) * x.std(dim=2, keepdim=True) * 0.01
    return x * gain + noise

def predict_tta(loader, model, n_tta=N_TTA, device=DEVICE):
    """Returns (hard_preds, true_labels, softmax_probs, ensemble_std)."""
    model.eval()
    all_probs, all_std, all_labels = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            # Accumulate over TTA passes
            pass_probs = torch.zeros(xb.shape[0], N_CLASSES, device=device)
            for _ in range(n_tta):
                pass_probs += F.softmax(model.predict(augment_batch(xb)), dim=1)
            pass_probs /= n_tta
            # Ensemble std (across members, no TTA) as disagreement signal
            member_probs = F.softmax(model.predict_all(xb), dim=2)  # (M, B, C)
            std          = member_probs.std(dim=0).mean(dim=1)       # (B,) scalar per sample
            all_probs.append(pass_probs.cpu())
            all_std.append(std.cpu())
            all_labels.extend(yb.numpy())
    probs = torch.cat(all_probs, 0).numpy()
    stds  = torch.cat(all_std, 0).numpy()
    return probs.argmax(1), np.array(all_labels), probs, stds


print("Loading best checkpoint for evaluation...")
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Epoch {ckpt['epoch']}  val_F1={ckpt['val_f1']:.4f}")


## 13. Conformal Prediction — Guaranteed Uncertainty Quantification

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SPLIT CONFORMAL PREDICTION
#
# Problem: after sim→real transfer, the model's softmax confidence values are
# unreliable — they may be overconfident on OOD real-rig data.
# Temperature scaling calibrated on synthetic data fails because calibration
# data ≠ deployment distribution (Tomani et al., CVPR 2021).
#
# Solution: conformal prediction provides a FINITE-SAMPLE, DISTRIBUTION-FREE
# guarantee:
#   P(true_class ∈ prediction_set) ≥ 1 − α   for any test point
# with NO distributional assumptions.
#
# How it works (split conformal):
#   1. Calibration set: compute nonconformity score s_i = 1 − p̂(y_i|x_i)
#      (probability assigned to the TRUE class by the model)
#   2. Compute q̂ = (1−α) quantile of {s_i} over calibration set
#   3. At inference: prediction_set = {c : 1 − p̂(c|x) ≤ q̂}
#      = all classes where the model's probability exceeds (1 − q̂)
#
# Guarantee: prediction_set contains the true class ≥ 90% of the time
# (at α=0.1), even under distribution shift, as long as calibration data
# has the same distribution as test data.
#
# For TopDrive AI: calibrate on real captures, deploy with sim→real guarantee.
# ─────────────────────────────────────────────────────────────────────────────

# Step 1 — Calibration scores
print("Computing conformal calibration scores on held-out calibration set...")
_, cal_labels, cal_probs, cal_stds = predict_tta(cal_loader, model, n_tta=1)

# Nonconformity score: 1 − (probability assigned to true class)
# Low score = model is confident and correct; high score = wrong or uncertain
cal_scores = 1.0 - cal_probs[np.arange(len(cal_labels)), cal_labels]
print(f"  Calibration samples : {len(cal_scores)}")
print(f"  Score distribution  : min={cal_scores.min():.4f}  "
      f"mean={cal_scores.mean():.4f}  "
      f"max={cal_scores.max():.4f}")

# Step 2 — Conformal threshold q̂
# The quantile level is adjusted for finite samples: ceil((n+1)(1−α))/n
n_cal     = len(cal_scores)
adjusted_q_level = np.ceil((n_cal + 1) * (1 - CP_ALPHA)) / n_cal
q_hat     = float(np.quantile(cal_scores, min(adjusted_q_level, 1.0)))
print(f"  q̂ (α={CP_ALPHA}) = {q_hat:.4f}  "
      f"→ threshold probability = {1 - q_hat:.4f}")

# Step 3 — Prediction function
def conformal_predict(probs: np.ndarray) -> list:
    """
    Given softmax probability vector, return the conformal prediction set.
    The set is guaranteed to contain the true class with probability ≥ 1−α.
    Smaller sets = more confident. Set of all 9 classes = complete uncertainty.
    """
    return [c for c in range(N_CLASSES) if (1 - probs[c]) <= q_hat]


def tier_classify(probs: np.ndarray, std: float) -> dict:
    """
    Three-tier deployment decision:
      Tier 1 — AUTO-CLASSIFY: small prediction set + low ensemble disagreement
      Tier 2 — ALERT OPERATOR: moderate uncertainty, show top-k predictions
      Tier 3 — ABSTAIN: high uncertainty, flag for manual diagnosis
    """
    pred_set     = conformal_predict(probs)
    hard_pred    = int(probs.argmax())
    confidence   = float(probs.max())
    set_size     = len(pred_set)
    entropy      = float(-np.sum(probs * np.log(probs + 1e-8)))
    max_entropy  = np.log(N_CLASSES)  # uniform distribution entropy

    if set_size == 1 and std < 0.05 and entropy < 0.5 * max_entropy:
        tier   = 1
        action = 'AUTO-CLASSIFY'
    elif set_size <= 3 and std < 0.15:
        tier   = 2
        action = 'ALERT OPERATOR'
    else:
        tier   = 3
        action = 'ABSTAIN — manual review'

    return {
        'tier'      : tier,
        'action'    : action,
        'prediction': CLASS_NAMES[hard_pred],
        'confidence': confidence,
        'pred_set'  : [CLASS_NAMES[c] for c in pred_set],
        'set_size'  : set_size,
        'ens_std'   : std,
        'entropy'   : entropy,
    }


# Evaluate tier distribution on calibration set
tier_counts = {1: 0, 2: 0, 3: 0}
for i in range(len(cal_labels)):
    t = tier_classify(cal_probs[i], cal_stds[i])
    tier_counts[t['tier']] += 1

total = sum(tier_counts.values())
print()
print("CALIBRATION SET TIER DISTRIBUTION")
print(f"  Tier 1 (auto-classify) : {tier_counts[1]:5d}  ({tier_counts[1]/total*100:.1f}%)")
print(f"  Tier 2 (alert operator): {tier_counts[2]:5d}  ({tier_counts[2]/total*100:.1f}%)")
print(f"  Tier 3 (abstain)       : {tier_counts[3]:5d}  ({tier_counts[3]/total*100:.1f}%)")
print()

# Verify coverage guarantee on calibration set
correct_in_set = sum(
    1 for i in range(len(cal_labels))
    if cal_labels[i] in conformal_predict(cal_probs[i])
)
empirical_cov = correct_in_set / total
print(f"Empirical calibration coverage: {empirical_cov:.4f}  (guarantee: ≥{1-CP_ALPHA:.2f})")
assert empirical_cov >= 1 - CP_ALPHA, "CP coverage violated — enlarge calibration set"
print("✓ Coverage guarantee verified")


## 14. Test Results + Exit Criteria

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, recall_score
import matplotlib.pyplot as plt
import seaborn as sns

print("Running TTA inference on test set...")
test_preds, test_labels, test_probs, test_stds = predict_tta(test_loader, model, n_tta=N_TTA)

macro_f1 = f1_score(test_labels, test_preds, average='macro', zero_division=0)
per_cls  = f1_score(test_labels, test_preds, average=None, zero_division=0, labels=list(range(9)))
recall_pc          = recall_score(test_labels, test_preds, average=None, zero_division=0, labels=list(range(9)))
avg_fault_recall   = recall_pc[1:].mean()

print()
print("=" * 65)
print("FINAL TEST RESULTS")
print("=" * 65)
print(classification_report(test_labels, test_preds,
      target_names=[CLASS_NAMES[i] for i in range(9)], digits=4, zero_division=0))

EXIT = {
    "Macro F1 ≥ 0.95"         : macro_f1         >= 0.95,
    "Min per-class F1 ≥ 0.85" : per_cls.min()    >= 0.85,
    "Normal recall ≥ 0.97"    : recall_pc[0]     >= 0.97,
    "Avg fault recall ≥ 0.90" : avg_fault_recall >= 0.90,
}
print("EXIT CRITERIA")
print("-" * 50)
all_pass = True
for crit, passed in EXIT.items():
    print(f"  {'✓ PASS' if passed else '✗ FAIL'}  {crit}")
    if not passed: all_pass = False
print()
print(f"OVERALL: {'✓ ALL CRITERIA MET — SHIP IT' if all_pass else '✗ CRITERIA NOT MET'}")
print(f"Macro F1 : {macro_f1:.4f}   Min class: {per_cls.min():.4f} ({CLASS_NAMES[int(per_cls.argmin())]})")

# CP coverage on test set
correct_test = sum(1 for i in range(len(test_labels))
                   if test_labels[i] in conformal_predict(test_probs[i]))
test_cov = correct_test / len(test_labels)
print()
print(f"Conformal prediction test coverage: {test_cov:.4f}  (guarantee: ≥{1-CP_ALPHA:.2f})")

avg_set_size = np.mean([len(conformal_predict(test_probs[i])) for i in range(len(test_labels))])
print(f"Mean prediction set size          : {avg_set_size:.2f}  (1.0 = perfect)")

tier_dist_test = {1: 0, 2: 0, 3: 0}
for i in range(len(test_labels)):
    t = tier_classify(test_probs[i], test_stds[i])
    tier_dist_test[t['tier']] += 1
total_t = sum(tier_dist_test.values())
print(f"\nTest set tier distribution:")
print(f"  Tier 1 (auto-classify) : {tier_dist_test[1]:5d}  ({tier_dist_test[1]/total_t*100:.1f}%)")
print(f"  Tier 2 (alert operator): {tier_dist_test[2]:5d}  ({tier_dist_test[2]/total_t*100:.1f}%)")
print(f"  Tier 3 (abstain)       : {tier_dist_test[3]:5d}  ({tier_dist_test[3]/total_t*100:.1f}%)")


In [ ]:
# ── Plots ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1. Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
labels = [CLASS_NAMES[i] for i in range(9)]
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, vmin=0, vmax=100, ax=axes[0,0])
axes[0,0].set_title(f'Confusion Matrix (%)  Macro F1={macro_f1:.4f}')
axes[0,0].set_xlabel('Predicted'); axes[0,0].set_ylabel('True')
plt.setp(axes[0,0].get_xticklabels(), rotation=45, ha='right')

# 2. Training curves
axes[0,1].plot(history['val_f1'], color='darkorange', label='Val Macro F1')
axes[0,1].plot(history['dann_loss'], color='steelblue', alpha=0.6, label='DANN loss')
axes[0,1].axhline(0.95, color='red', ls='--', alpha=0.7, label='Target 0.95')
axes[0,1].set_title('Training Progress'); axes[0,1].set_xlabel('Epoch')
axes[0,1].legend(fontsize=8); axes[0,1].grid(alpha=0.3)

# 3. Per-class F1
per_class_arr = np.array(history['per_class'])
for c in range(9):
    axes[1,0].plot(per_class_arr[:, c], label=CLASS_NAMES[c][:10], alpha=0.7)
axes[1,0].axhline(0.85, color='red', ls='--', alpha=0.5, label='0.85 floor')
axes[1,0].set_title('Per-Class F1 (val)'); axes[1,0].set_xlabel('Epoch')
axes[1,0].legend(fontsize=6, ncol=2); axes[1,0].grid(alpha=0.3)

# 4. Conformal prediction set size distribution
set_sizes = [len(conformal_predict(test_probs[i])) for i in range(len(test_labels))]
axes[1,1].hist(set_sizes, bins=range(1, 11), align='left', rwidth=0.7, color='teal')
axes[1,1].set_title(f'CP Prediction Set Sizes  (avg={avg_set_size:.2f})')
axes[1,1].set_xlabel('Prediction set size'); axes[1,1].set_ylabel('Count')
axes[1,1].axvline(avg_set_size, color='red', ls='--', label=f'Mean={avg_set_size:.2f}')
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'results_v3.png', dpi=150)
plt.show()


## 15. Export — ONNX + Inference Package

In [ ]:
import json, datetime, pickle

# ── Save conformal threshold for deployment ───────────────────────────────
cp_config = {
    'q_hat'       : float(q_hat),
    'alpha'       : CP_ALPHA,
    'n_cal'       : int(n_cal),
    'class_names' : CLASS_NAMES,
    'tier_thresholds': {
        'tier1_max_set_size'  : 1,
        'tier1_max_std'       : 0.05,
        'tier1_max_entropy_r' : 0.5,
        'tier2_max_set_size'  : 3,
        'tier2_max_std'       : 0.15,
    }
}
with open(MODEL_DIR / 'cp_config_v3.json', 'w') as f:
    json.dump(cp_config, f, indent=2)
print(f"CP config → {MODEL_DIR/'cp_config_v3.json'}")

# ── ONNX export ───────────────────────────────────────────────────────────
try:
    dummy  = torch.zeros(1, n_channels, seq_len).to(DEVICE)
    onnx_p = MODEL_DIR / 'topdrive_v3.onnx'
    torch.onnx.export(
        model, dummy, str(onnx_p),
        input_names=['sensor_window'], output_names=['fault_logits'],
        dynamic_axes={'sensor_window': {0: 'batch_size'}}, opset_version=17,
    )
    print(f"ONNX → {onnx_p}")
except Exception as e:
    print(f"ONNX export error (non-critical): {e}")

# ── Training report ───────────────────────────────────────────────────────
report = {
    "generated_at"    : datetime.datetime.utcnow().isoformat(),
    "macro_f1_tta"    : round(float(macro_f1), 4),
    "cp_coverage"     : round(float(test_cov), 4),
    "cp_q_hat"        : round(float(q_hat), 4),
    "avg_set_size"    : round(float(avg_set_size), 4),
    "tier_distribution": {k: int(v) for k, v in tier_dist_test.items()},
    "per_class_f1"    : {CLASS_NAMES[i]: round(float(per_cls[i]), 4) for i in range(9)},
    "exit_criteria"   : {k: bool(v) for k, v in EXIT.items()},
    "best_val_f1"     : round(float(best_val_f1), 4),
    "hyperparams"     : {
        "window_size": WINDOW_SIZE, "stride": STRIDE, "batch_size": BATCH_SIZE,
        "n_epochs": N_EPOCHS, "lr": LR, "focal_gamma": FOCAL_GAMMA,
        "ensemble_size": ENSEMBLE_SIZE, "n_tta": N_TTA,
        "dann_lambda_max": DANN_LAMBDA_MAX, "dann_warmup": DANN_WARMUP,
        "l2sp_alpha": L2SP_ALPHA, "cp_alpha": CP_ALPHA,
        "model": "DANNEnsemble(MultiScaleInceptionTime+GRL) v3",
    },
}
with open(MODEL_DIR / 'report_v3.json', 'w') as f:
    json.dump(report, f, indent=2)
print(f"Report → {MODEL_DIR/'report_v3.json'}")

print()
print("=" * 55)
print("v3 TRAINING COMPLETE")
print(f"  Macro F1 (TTA)     : {macro_f1:.4f}")
print(f"  CP coverage        : {test_cov:.4f}  (guarantee ≥ {1-CP_ALPHA:.2f})")
print(f"  Mean set size      : {avg_set_size:.2f}")
print(f"  Best val F1        : {best_val_f1:.4f}")
print(f"  Checkpoint         : {ckpt_path}")
print()
print("Next steps:")
print("  1. Set REAL_DIR in config and re-run domain gap quantification (NB1)")
print("  2. Set REAL_LABELED_DIR and run L2-SP fine-tuning (cell 10)")
print("  3. Run apply_adabn() per-machine before deployment on each rig")
print("=" * 55)
